# 04 — Functional gradients (BrainSpace)


In [1]:
import os
from pathlib import Path
import yaml

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent
elif not (PROJECT_ROOT / "config.yaml").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent

with open(PROJECT_ROOT / "config.yaml", encoding="utf-8") as f:
    CONFIG = yaml.safe_load(f)

    DATA_MODE = CONFIG.get("data_mode", "example")
    N_STATES = CONFIG["n_states"]
    N_TIMEPOINTS = CONFIG["n_timepoints"]
    N_SUBJECTS = CONFIG["n_subjects"]
    METHOD = CONFIG["method"]
    USE_GAMMA = CONFIG["use_gamma"]
    MIN_EFF_N = CONFIG["min_eff_n"]
    MIN_HARD_N = CONFIG["min_hard_n"]
    GRAD = CONFIG["gradient"]

    DATA_ROOT = PROJECT_ROOT / (CONFIG["paths"]["example_root"] if DATA_MODE == "example" else CONFIG["paths"]["study_data_root"])
    OUTPUT_ROOT = PROJECT_ROOT / CONFIG["paths"]["output_root"] / DATA_MODE
    OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)

    print(f"project={PROJECT_ROOT} mode={DATA_MODE} out={OUTPUT_ROOT}")

    import os
    for k in ("OPENBLAS_NUM_THREADS","OMP_NUM_THREADS","MKL_NUM_THREADS","NUMEXPR_NUM_THREADS","VECLIB_MAXIMUM_THREADS"):
        os.environ.setdefault(k,"1")

project=d:\文件同步\文件\数据分析结果\data_code_figure\gradient_dispersion_mediate_moderate\gradient_analysis mode=example out=d:\文件同步\文件\数据分析结果\data_code_figure\gradient_dispersion_mediate_moderate\gradient_analysis\results\example


In [2]:
# Functional gradient metrics via BrainSpace
# Notebook workflow; optional multiprocessing

# Limit BLAS threads before numpy (if this cell runs without cell 0)
import os
os.environ.setdefault("OPENBLAS_NUM_THREADS", "1")
os.environ.setdefault("OMP_NUM_THREADS", "1")
os.environ.setdefault("MKL_NUM_THREADS", "1")
os.environ.setdefault("NUMEXPR_NUM_THREADS", "1")
os.environ.setdefault("VECLIB_MAXIMUM_THREADS", "1")

import numpy as np
import pandas as pd
from scipy import stats
import warnings
import multiprocessing as mp
from functools import partial
from tqdm import tqdm
import time
import os
import matplotlib.pyplot as plt
import seaborn as sns
warnings.filterwarnings('ignore')

# BrainSpace import
try:
    from brainspace.gradient import GradientMaps
    print("BrainSpace imported")
except ImportError:
    raise ImportError("Install BrainSpace: pip install brainspace")

    # Reproducibility
    np.random.seed(42)
    plt.rcParams['figure.figsize'] = (12, 8)
    plt.rcParams['font.size'] = 10
    sns.set_style("whitegrid")


def calculate_gradient_metrics(FC_matrix, n_components=10, approach='dm', kernel='normalized_angle',
alignment='procrustes', sparsity_percentile=10, random_state=42):
    """
    Compute functional gradient metrics with BrainSpace

    Parameters:
    FC_matrix: functional connectivity matrices (n_regions, n_regions)
    n_components: Number of gradient components
    approach: Gradient approach ('dm', 'le', 'pca')
    kernel: Kernel (default normalized_angle)
    alignment: Alignment (default Procrustes)
    sparsity_percentile: Top N% edges per region (default 10%)
    random_state: Random seed

    Returns:
    dict: Dict with gradient metrics and scores
    """
    # Validate input
    if FC_matrix is None:
        return {
        'gradient_scores': None,
        'gradient_range': np.nan,
        'gradient_dispersion': np.nan,
        'gradients': None
        }

        if np.all(np.isnan(FC_matrix)):
            return {
            'gradient_scores': np.full(FC_matrix.shape[0], np.nan),
            'gradient_range': np.nan,
            'gradient_dispersion': np.nan,
            'gradients': None
            }

            # Replace NaN with 0
            FC_clean = np.nan_to_num(FC_matrix, nan=0.0)

            # Symmetrize
            FC_clean = (FC_clean + FC_clean.T) / 2

            # Zero diagonal
            np.fill_diagonal(FC_clean, 0)

            # Check usable matrix
            if np.all(FC_clean == 0) or FC_clean.shape[0] < 3:
                return {
                'gradient_scores': np.full(FC_clean.shape[0], np.nan),
                'gradient_range': np.nan,
                'gradient_dispersion': np.nan,
                'gradients': None
                }

                # Sparsify:
                # 1. Zero negative edges
                # 2. Keep top percentile of positive edges per region
                FC_sparse = FC_clean.copy()

                # Step 1: zero negatives
                FC_sparse[FC_sparse < 0] = 0

                n_regions = FC_sparse.shape[0]

                # Step 2: keep top percentile per region
                for i in range(n_regions):
                    # Original row weights for ranking (exclude diagonal)
                    row_connections_original = FC_clean[i, :].copy()
                    row_connections_original[i] = -np.inf # Exclude self-connection from ranking

                    # Number of edges to keep (of n_regions - 1)
                    # n_regions - 1 off-diagonal slots
                    n_total_connections = n_regions - 1
                    n_keep = max(1, int(np.ceil(n_total_connections * sparsity_percentile / 100)))

                    # Rank by raw strength (not absolute value)
                    row_connections_original[i] = -np.inf # Keep diagonal excluded

                    # Indices of top-percentile edges
                    top_indices = np.argsort(row_connections_original)[-n_keep:]

                    # Mask: top percentile and positive only
                    mask = np.zeros(n_regions, dtype=bool)
                    for idx in top_indices:
                        if FC_sparse[i, idx] > 0: # Positive edges only
                            mask[idx] = True
                            mask[i] = False # Diagonal False

                            # Zero edges outside mask
                            FC_sparse[i, ~mask] = 0

                            # Re-symmetrize sparse matrix
                            FC_sparse = (FC_sparse + FC_sparse.T) / 2

                            # Check edge count
                            # Count nonzeros off-diagonal
                            np.fill_diagonal(FC_sparse, 0) # Zero diagonal
                            n_nonzero = np.count_nonzero(FC_sparse)

                            # Minimum edges for BrainSpace
                            # BrainSpace needs some edges but not full connectivity
                            # Small graphs: at least n_regions - 1 edges
                            # Larger graphs: more edges required
                            if n_regions < 5:
                                min_connections = max(1, n_regions - 1)
                            elif n_regions < 10:
                                min_connections = max(2, n_regions - 1)
                            else:
                                min_connections = max(3, int(n_regions * 0.5)) # At least half the nodes connected

                                if n_nonzero < min_connections or np.all(FC_sparse == 0):
                                    # Insufficient edges: return NaN quietly
                                    return {
                                    'gradient_scores': np.full(FC_sparse.shape[0], np.nan),
                                    'gradient_range': np.nan,
                                    'gradient_dispersion': np.nan,
                                    'gradients': None
                                    }

                                    try:
                                        # Fit GradientMaps
                                        gm = GradientMaps(
                                        n_components=n_components,
                                        approach=approach,
                                        kernel=kernel,
                                        alignment=alignment,
                                        random_state=random_state
                                        )

                                        # Fit gradients
                                        gm.fit(FC_sparse)

                                        # First gradient component
                                        if gm.gradients_ is not None and gm.gradients_.shape[1] > 0:
                                            gradient_scores = gm.gradients_[:, 0] # Primary gradient scores

                                            # Global gradient range
                                            gradient_range = np.max(gradient_scores) - np.min(gradient_scores)

                                            # Global gradient dispersion (std)
                                            gradient_dispersion = np.std(gradient_scores)

                                            return {
                                            'gradient_scores': gradient_scores,
                                            'gradient_range': gradient_range,
                                            'gradient_dispersion': gradient_dispersion,
                                            'gradients': gm.gradients_
                                            }
                                        else:
                                            return {
                                            'gradient_scores': np.full(FC_sparse.shape[0], np.nan),
                                            'gradient_range': np.nan,
                                            'gradient_dispersion': np.nan,
                                            'gradients': None
                                            }
                                    except Exception as e:
                                        # On failure return NaN without noisy logging
                                        # Expected when a network lacks sufficient edges
                                        return {
                                        'gradient_scores': np.full(FC_sparse.shape[0], np.nan),
                                        'gradient_range': np.nan,
                                        'gradient_dispersion': np.nan,
                                        'gradients': None
                                        }

                                        print("Gradient metric helpers loaded (BrainSpace).")

                                        # ── Panel A G1/G2 export helpers (Figure 5) ──
                                        import os

                                        # Schaefer-2018 7-network names (keep as-is in exported CSV).
                                        # Optional alias map only for Figure 5 panel colours — NOT applied to saved G1/G2.
                                        NET_CANONICAL = {
                                        "Vis": "Vis",
                                        "SomMot": "SomMot",
                                        "DorsAttn": "DorsAttn",
                                        "SalVentAttn": "SalVentAttn",
                                        "Limbic": "Limbic",
                                        "Cont": "Cont",
                                        "Default": "Default"
                                        }

                                        PANEL_A_G12_FILENAME = "weighted_average_gradient1_gradient2_parcels_{k}states_{approach}.csv"


def canonical_network(name):
    """Map Schaefer / Yeo label token to Figure 5 network key."""
    if not isinstance(name, str):
        name = str(name)
        for key, canon in NET_CANONICAL.items():
            if key in name:
                return canon
                return "Cont"


def _orient_unimodal_to_transmodal(g1, g2, networks):
    """Flip G1/G2 if visual > default on G1 (Margulies convention)."""
    g1 = np.asarray(g1, dtype=float).copy()
    g2 = np.asarray(g2, dtype=float).copy()
    vis = g1[networks == "Vis"]
    default = g1[networks == "Default"]
    if len(vis) and len(default) and np.nanmedian(vis) > np.nanmedian(default):
        g1, g2 = -g1, -g2
        return g1, g2


def parcel_g12_filename(k, approach="dm"):
    return PANEL_A_G12_FILENAME.format(k=k, approach=approach)


def aggregate_parcel_gradients_from_subjects(subject_gradients, network_labels, n_rois=None):
    """Median G1/G2 per parcel across subjects (reuses per-subject BrainSpace fits)."""
    if not subject_gradients:
        raise ValueError("No subject gradients to aggregate.")
    network_labels = np.asarray(network_labels)
    n_labels = len(network_labels)
    n_grad = subject_gradients[0][0].shape[0]
    if n_rois is not None and n_rois != n_labels:
        raise ValueError(
    f"ROI label count ({n_labels}) != FC/gradient ROI count ({n_rois}). "
    "Re-run the mask/network-filter cell and FC extraction with the same select_networks."
    )
    if n_grad != n_labels:
        raise ValueError(
    f"Gradient length ({n_grad}) != network label count ({n_labels}). "
    "Pass select_networks in the same column order as weighted_avg_FC."
    )
    networks_raw = np.asarray([str(n) for n in network_labels], dtype=object)
    g1_stack, g2_stack = [], []
    for g1, g2 in subject_gradients:
        g1 = np.asarray(g1, dtype=float)
        g2 = np.asarray(g2, dtype=float)
        g1, g2 = _orient_unimodal_to_transmodal(g1, g2, networks_raw)
        g1_stack.append(g1)
        g2_stack.append(g2)
        g1_med = np.nanmedian(np.stack(g1_stack, axis=0), axis=0)
        g2_med = np.nanmedian(np.stack(g2_stack, axis=0), axis=0)
        rows = []
        for i, net in enumerate(networks_raw):
            rows.append({
            "parcel": i,
            "network": net,
            "g1": float(g1_med[i]),
            "g2": float(g2_med[i]),
            "is_sn": net == "SalVentAttn",
            })
            df = pd.DataFrame(rows)
            df.attrs["n_subjects"] = len(subject_gradients)
            df.attrs["n_rois"] = n_labels
            return df


def save_parcel_g12_csv(df, out_path):
    out_dir = os.path.dirname(out_path)
    if out_dir:
        os.makedirs(out_dir, exist_ok=True)
        df.to_csv(out_path, index=False)
        return out_path

        print("Panel A G1/G2 export helpers loaded.")

BrainSpace imported
Gradient metric helpers loaded (BrainSpace).
Panel A G1/G2 export helpers loaded.


In [3]:
# Per-subject weighted-average FC gradient metrics
def process_single_subject_weighted_avg(
    subj_data,
    sorted_networks,
    R,
    n_components=10,
    approach='dm',
    kernel='normalized_angle',
    alignment='procrustes',
    sparsity_percentile=10,
):
    """
    Gradient metrics for one subject's weighted-average FC

    Network metrics are extracted from the whole-brain embedding, not separate subgraph gradients
    """
    subj, FC = subj_data
    subj_results = []

    if FC is None or FC.shape[0] != R:
        return subj_results, None

    if np.all(np.isnan(FC)):
        print(f"Warning: subject {subj} weighted-average FC is all NaN; skipping")
        return subj_results, None

    global_gradient_metrics = calculate_gradient_metrics(
        FC,
        n_components=n_components,
        approach=approach,
        kernel=kernel,
        alignment=alignment,
        sparsity_percentile=sparsity_percentile,
    )

    result_row = {'subject': subj}
    result_row['global_gradient_range'] = global_gradient_metrics['gradient_range']
    result_row['global_gradient_dispersion'] = global_gradient_metrics['gradient_dispersion']

    network_metrics = {}
    if sorted_networks is not None and global_gradient_metrics['gradient_scores'] is not None:
        unique_networks = np.unique(sorted_networks)
        global_gradient_scores = global_gradient_metrics['gradient_scores']

        for net in unique_networks:
            net_indices = np.where(sorted_networks == net)[0]
            if len(net_indices) < 3:
                continue

            net_gradient_scores = global_gradient_scores[net_indices]
            valid_scores = net_gradient_scores[np.isfinite(net_gradient_scores)]

            if len(valid_scores) > 0:
                net_gradient_range = np.max(valid_scores) - np.min(valid_scores)
                net_gradient_dispersion = np.std(valid_scores)
                net_gradient_score_mean = np.mean(valid_scores)
                network_metrics[f'net_{net}_gradient_range'] = net_gradient_range
                network_metrics[f'net_{net}_gradient_dispersion'] = net_gradient_dispersion
                network_metrics[f'net_{net}_gradient_score_mean'] = net_gradient_score_mean
            else:
                network_metrics[f'net_{net}_gradient_range'] = np.nan
                network_metrics[f'net_{net}_gradient_dispersion'] = np.nan
                network_metrics[f'net_{net}_gradient_score_mean'] = np.nan

        result_row.update(network_metrics)

        if len(unique_networks) >= 2:
            from itertools import combinations

            hierarchy_span_metrics = {}
            net_mean_scores = {}
            for net in unique_networks:
                net_indices = np.where(sorted_networks == net)[0]
                if len(net_indices) >= 3:
                    net_gradient_scores = global_gradient_scores[net_indices]
                    valid_scores = net_gradient_scores[np.isfinite(net_gradient_scores)]
                    if len(valid_scores) > 0:
                        net_mean_scores[net] = np.mean(valid_scores)
                    else:
                        net_mean_scores[net] = np.nan
                else:
                    net_mean_scores[net] = np.nan

            for net1, net2 in combinations(unique_networks, 2):
                if net1 in net_mean_scores and net2 in net_mean_scores:
                    score1 = net_mean_scores[net1]
                    score2 = net_mean_scores[net2]
                    if np.isfinite(score1) and np.isfinite(score2):
                        hierarchy_span_metrics[f'hierarchy_span_{net1}_{net2}'] = abs(score1 - score2)
                    else:
                        hierarchy_span_metrics[f'hierarchy_span_{net1}_{net2}'] = np.nan
                else:
                    hierarchy_span_metrics[f'hierarchy_span_{net1}_{net2}'] = np.nan

            result_row.update(hierarchy_span_metrics)

    elif sorted_networks is not None:
        network_metrics = {}
        unique_networks = np.unique(sorted_networks)
        for net in unique_networks:
            net_indices = np.where(sorted_networks == net)[0]
            if len(net_indices) >= 3:
                network_metrics[f'net_{net}_gradient_range'] = np.nan
                network_metrics[f'net_{net}_gradient_dispersion'] = np.nan
                network_metrics[f'net_{net}_gradient_score_mean'] = np.nan
        result_row.update(network_metrics)

        if len(unique_networks) >= 2:
            from itertools import combinations

            hierarchy_span_metrics = {}
            for net1, net2 in combinations(unique_networks, 2):
                hierarchy_span_metrics[f'hierarchy_span_{net1}_{net2}'] = np.nan
            result_row.update(hierarchy_span_metrics)

    subj_results.append(result_row)

    grad_payload = None
    grads = global_gradient_metrics.get("gradients")
    if grads is not None and grads.shape[0] == R and grads.shape[1] >= 1:
        g1 = np.asarray(grads[:, 0], dtype=float)
        g2 = np.asarray(grads[:, 1], dtype=float) if grads.shape[1] > 1 else np.zeros_like(g1)
        grad_payload = (g1, g2)

    return subj_results, grad_payload


print("Per-subject gradient processor loaded.")

Per-subject gradient processor loaded.


In [4]:
# Batch gradient metrics for weighted-average FC

def _blas_thread_guard():
    """Worker init: one BLAS thread per process when using multiprocessing."""
    import os
    for key in (
    "OPENBLAS_NUM_THREADS", "OMP_NUM_THREADS", "MKL_NUM_THREADS",
    "NUMEXPR_NUM_THREADS", "VECLIB_MAXIMUM_THREADS",
    ):
        os.environ[key] = "1"

def run_gradient_metrics(weighted_avg_FC, sorted_networks=None, rt_vars_df=None,
use_parallel=True, n_jobs=None, save_dir="results",
n_components=10, approach='dm', kernel='normalized_angle',
alignment='procrustes', sparsity_percentile=10):
    """
    Batch gradient metrics for weighted-average FC

    Parameters:
    -----------
    weighted_avg_FC : dict
    Per-subject weighted-average functional connectivity matrices
    sorted_networks : array
    Network label vector
    rt_vars_df : DataFrame
    Behavioral covariates (optional)
    use_parallel : bool
    Use multiprocessing
    n_jobs : int
    Worker count
    save_dir : str
    Output directory
    n_components : int
    Number of gradient components (default 10)
    approach : str
    Gradient approach ('dm', 'le', 'pca') (default 'dm')
    kernel : str
    N_STATESernel (default normalized_angle)
    alignment : str
    Alignment (default Procrustes)
    sparsity_percentile : int
    Top N% edges per region (default 10%)

    Returns:
    --------
    tuple
    (results_df, correlation_df, significant_df)
    """
    import time
    import os
    import multiprocessing as mp
    from functools import partial
    from tqdm import tqdm
    import pandas as pd
    import numpy as np
    from scipy.stats import pearsonr
    from statsmodels.stats.multitest import multipletests
    print("=== Weighted-average FC gradient analysis ===")
    print(f"Gradient approach: {approach}")
    print(f"Components: {n_components}")
    print(f"N_STATESernel: {kernel}")
    print(f"Alignment: {alignment}")
    print(f"Sparsity: top {sparsity_percentile}% edges per region")
    print(f"Parallel: {use_parallel}")

    # Timing
    start_time = time.time()

    # Subjects and ROI count
    valid_subjects = [subj for subj, fc in weighted_avg_FC.items() if fc is not None]
    if len(valid_subjects) == 0:
        raise ValueError("No valid weighted-average FC matrices")

    any_subj = valid_subjects[0]
    R = weighted_avg_FC[any_subj].shape[0]
    n_subjects = len(valid_subjects)

    print(f"ROIs: {R}, valid subjects: {n_subjects}")
    if sorted_networks is not None and len(sorted_networks) != R:
        raise ValueError(
    f"Network labels ({len(sorted_networks)}) != FC matrix ROIs ({R}). "
    "Use select_networks in the same order as Y / weighted_avg_FC columns."
    )

    # Worker count
    if n_jobs is None:
        n_jobs = min(mp.cpu_count(), n_subjects)
        # Cap workers: each process uses BLAS; too many × 48 threads hits RLIMIT_NPROC
        max_workers = int(os.environ.get("GRADIENT_MP_MAX_WORN_STATESERS", "48"))
        n_jobs = max(1, min(n_jobs, max_workers))
        print(f"Workers: {n_jobs} (BLAS threads=1 per process)")

        # Subject list
        subj_data_list = [(subj, weighted_avg_FC[subj]) for subj in valid_subjects]

        # Partial for workers
        process_func = partial(process_single_subject_weighted_avg,
        sorted_networks=sorted_networks,
        R=R,
        n_components=n_components,
        approach=approach,
        kernel=kernel,
        alignment=alignment,
        sparsity_percentile=sparsity_percentile)

        # Run per subject
        all_results = []
        subject_gradients = []
        start_time_parallel = time.time()

        if use_parallel:
            with mp.Pool(processes=n_jobs, initializer=_blas_thread_guard) as pool:
                results = list(tqdm(
                pool.imap(process_func, subj_data_list),
                total=len(subj_data_list),
                desc="subjects"
                ))

                # Collect results
                for subj_result, grad_payload in results:
                    all_results.extend(subj_result)
                    if grad_payload is not None:
                        subject_gradients.append(grad_payload)
                    else:
                        # Serial fallback
                        for subj_data in tqdm(subj_data_list, desc="subjects"):
                            subj_result, grad_payload = process_func(subj_data)
                            all_results.extend(subj_result)
                            if grad_payload is not None:
                                subject_gradients.append(grad_payload)

                                end_time_parallel = time.time()

                                # Export parcel G1/G2 (group median across subjects) for Figure 5 Panel A
                                if subject_gradients and sorted_networks is not None:
                                    try:
                                        g12_df = aggregate_parcel_gradients_from_subjects(
                                        subject_gradients, sorted_networks, n_rois=R,
                                        )
                                        g12_name = parcel_g12_filename(N_STATES, approach)
                                        g12_path = os.path.join(save_dir, g12_name)
                                        save_parcel_g12_csv(g12_df, g12_path)
                                        print(
                                        f"Parcel G1/G2 embedding saved: {g12_path} "
                                        f"(n_parcels={len(g12_df)}, n_subjects={g12_df.attrs.get('n_subjects')})"
                                        )
                                    except Exception as exc:
                                        print(f"Warning: parcel G1/G2 export failed: {exc}")
                            elif not subject_gradients:
                                print("Warning: no subject gradients collected; skipping G1/G2 export.")

                                print(f"Gradient metrics done in {end_time_parallel - start_time_parallel:.2f} s")

                                # Ensure output dir
                                os.makedirs(save_dir, exist_ok=True)

                                # Save CSV
                                results_df = pd.DataFrame(all_results)
                                timestamp = time.strftime("%Y%m%d_%H%M%S")
                                filename = f"weighted_average_gradient_metrics_1000parcels_{N_STATES}states_{METHOD}_{approach}.csv"
                                filepath = os.path.join(save_dir, filename)
                                results_df.to_csv(filepath, index=False)
                                print(f"Saved gradient metrics: {filepath}")


                                return results_df

In [5]:
import numpy as np
import pandas as pd

fc = np.load(OUTPUT_ROOT / "03_weighted_fc" / "weighted_average_FC.npz", allow_pickle=True)["weighted_avg_FC"].item()
sorted_networks = pd.read_csv(OUTPUT_ROOT / "01_roi" / "sorted_networks.csv")["network"].values
save_dir = OUTPUT_ROOT / "04_gradients"
save_dir.mkdir(parents=True, exist_ok=True)

results_df = run_gradient_metrics(
weighted_avg_FC=fc, sorted_networks=sorted_networks, rt_vars_df=None,
use_parallel=GRAD["use_parallel"], save_dir=str(save_dir),
n_components=GRAD["n_components"], approach=GRAD["approach"],
kernel=GRAD["kernel"], alignment=GRAD["alignment"],
sparsity_percentile=GRAD["sparsity_percentile"],
)
print(results_df.filter(regex="global_").head())

=== Weighted-average FC gradient analysis ===
Gradient approach: dm
Components: 10
N_STATESernel: normalized_angle
Alignment: procrustes
Sparsity: top 10% edges per region
Parallel: False
ROIs: 35, valid subjects: 5
Workers: 5 (BLAS threads=1 per process)


subjects: 0%| | 0/5 [00:00<?, ?it/s]

subjects: 100%|██████████| 5/5 [00:00<00:00, 476.89it/s]

Parcel G1/G2 embedding saved: d:\文件同步\文件\数据分析结果\data_code_figure\gradient_dispersion_mediate_moderate\gradient_analysis\results\example\04_gradients\weighted_average_gradient1_gradient2_parcels_3states_dm.csv (n_parcels=35, n_subjects=5)
Gradient metrics done in 0.01 s
Saved gradient metrics: d:\文件同步\文件\数据分析结果\data_code_figure\gradient_dispersion_mediate_moderate\gradient_analysis\results\example\04_gradients\weighted_average_gradient_metrics_1000parcels_3states_merged_all_statistical_dm.csv
 global_gradient_range global_gradient_dispersion
0 0.300552 0.087038
1 0.316740 0.082861
2 0.278721 0.088221
3 0.307114 0.080139
4 0.335635 0.083349
